# Snowflake — VKG (Ontology) Semantic-Layer Build

This notebook builds a **VKG (virtual knowledge graph / OWL) semantic layer over
live Snowflake tables**, reached through the deployed **Athena federated Snowflake
connector**, so the **ontology_query** agent can answer natural-language questions
against them. It is step **1 of 2**: notebook `18_snowflake_query_eval.ipynb`
queries the layer this notebook builds.

Structurally it mirrors `13_cwd_semantic_layer.ipynb` (and `11_bird_…`): seed a
DynamoDB VKG config, invoke the deployed **ontology generation** agent
(`OntologyRuntimeArn`), block on the DynamoDB build status, then score the
generated ontology **server-side** with an `OntologyQuality` LLM-as-Judge +
`Builtin.GoalSuccessRate` via AgentCore Batch Evaluations.

### What is different from the CWD / BIRD notebooks

| | CWD / BIRD | Here |
|:--|:--|:--|
| Source of table coordinates | `data/eval/benchmarks/*/manifest.json` written by the ingest driver | **live Athena** `list_table_metadata` on the connector catalog — nothing is ingested, so there is no manifest |
| Catalog | S3 Tables federated catalog (`s3tablescatalog/<bucket>`) | **Lambda-backed connector catalog** (`snowflake_<database>`) |
| Where the data lives | Iceberg tables in this account | **Snowflake**. Athena only holds the connector registration; there is **no Glue database and no Glue table** for these tables at all |
| Schema-introspection guard ground truth | Glue `get_tables` | **Athena** `list_table_metadata` (Glue has nothing to return) |

> **Two connector-specific traps this notebook exists to exercise.** Athena rejects
> `DESCRIBE "db"."table"` on *any* Lambda-backed catalog at submit time
> (`InvalidRequestException: no viable alternative at input 'DESCRIBE "FHIR_R4"'`),
> so the ontology agent must route federated catalogs to a `SELECT * … LIMIT 0`
> metadata read; and the Glue-enrichment step must *skip* a federated table rather
> than fail on `EntityNotFound`. Both are handled in
> `agents/ontology_agent/main.py` — if the deployed container predates that fix,
> the build completes with a guessed ontology and the guard in §5 fails it.

### Layer-id convention (IMPORTANT — notebook 18 depends on it)
The Snowflake database gets ONE VKG layer under a **deterministic** DynamoDB `id`:

> **`LAYER_ID = f"vkg-snowflake_{SNOWFLAKE_SCHEMA.lower()}"`** → with the default
> schema `FHIR_R4` that is **`vkg-snowflake_fhir_r4`**.

Notebook 18 derives the id from the **same two constants** (`SNOWFLAKE_CATALOG` /
`SNOWFLAKE_SCHEMA`, both env-overridable) so the two notebooks cannot drift.
Re-running seeds `status='pending'` over the same id and rebuilds in place.

### Prerequisites
- The agentcore stack deployed **with the connector enabled**:
  `npx cdk deploy -c enableSnowflakeConnector=true` from `cdk/`, plus
  `npm run sync:snowflake-secret` and `npm run set:snowflake-casing` (see
  `cdk/CDK_README.md`). Raw Athena must already be able to read the tables —
  verify with `SELECT count(*) FROM "FHIR_R4"."PATIENTS"` in the console before
  spending a build here.
- `bedrock-agentcore>=1.11.0` and `bedrock-agentcore-starter-toolkit==0.3.9`.


In [1]:
# Prereq (uncomment if not already installed):
# !pip install 'bedrock-agentcore>=1.11.0' bedrock-agentcore-starter-toolkit==0.3.9 boto3 pandas python-dotenv --quiet

In [2]:
# Verify the AgentCore Evaluation SDK is available
try:
    import bedrock_agentcore  # noqa: F401
    from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
        BatchEvaluationRunner,  # noqa: F401
    )
    print("✓ bedrock-agentcore installed:",
          getattr(bedrock_agentcore, "__version__", "version unknown"))
    print("✓ BatchEvaluationRunner import OK")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("  Install with: pip install 'bedrock-agentcore>=1.11.0'")
    raise

✓ bedrock-agentcore installed: version unknown
✓ BatchEvaluationRunner import OK


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


## 1. Setup & Credentials

In [4]:
import os
import sys  # noqa: F401  (kept for parity with sibling eval notebooks)
import json
import time
import uuid
import pandas as pd
import boto3
from botocore.config import Config
from datetime import datetime, timezone

from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
from bedrock_agentcore.evaluation.runner.dataset_types import (
    Dataset, PredefinedScenario, Turn,
)
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput,
)

print("✓ Imports successful")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

✓ Imports successful


In [5]:
# Load environment variables from .env file (sourced from cdk/cdk-outputs-agentcore.json)
from dotenv import load_dotenv, set_key

ENV_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(), '.env')
load_dotenv(dotenv_path=ENV_FILE, override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'default'))
region = session.region_name or 'us-east-1'

# ── CloudFormation output resolver ──
# For any env var still set to 'XXX' (no .env or .env missing the key), resolve from
# live CloudFormation stack outputs. Stacks may be deployed with a 'dev' suffix.
def _get_stack_outputs(stack_suffix: str) -> dict:
    """Return {OutputKey: OutputValue} for '{PROJECT_NAME}[-dev]-{suffix}', or {} if absent."""
    cfn = session.client('cloudformation', region_name=region)
    for name in (f'{PROJECT_NAME}-dev-{stack_suffix}', f'{PROJECT_NAME}-{stack_suffix}'):
        try:
            resp = cfn.describe_stacks(StackName=name)
            return {o['OutputKey']: o['OutputValue'] for o in resp['Stacks'][0].get('Outputs', [])}
        except Exception:
            continue
    return {}

_CFN_MAP = {
    'ARTIFACTS_BUCKET':        ('data-lake',  'ArtifactsBucketName'),
    'ATHENA_WORKGROUP':        ('athena',     'AthenaWorkgroupName'),
    'ONTOLOGY_METADATA_TABLE': ('dynamodb',   'MetadataTableName'),
    'KNOWLEDGE_BASE_ID':       ('bedrock-kb', 'OntologyPatternsKbId'),
}
_cfn_cache = {}
_resolved_from_cfn = []
for env_var, (stack, output_key) in _CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)
if _resolved_from_cfn:
    for env_var in _resolved_from_cfn:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_resolved_from_cfn)} CFN-resolved variable(s) to .env: {', '.join(_resolved_from_cfn)}")

print("\n✓ Environment variables configured:")
for key in ['ARTIFACTS_BUCKET', 'ATHENA_WORKGROUP', 'ONTOLOGY_METADATA_TABLE', 'KNOWLEDGE_BASE_ID']:
    print(f"  {key} = {_mask_acct(str(os.environ.get(key, '(not set)')))}")


✓ Environment variables configured:
  ARTIFACTS_BUCKET = semantic-layer-dev-artifacts-********4087
  ATHENA_WORKGROUP = semantic-layer-dev-workgroup
  ONTOLOGY_METADATA_TABLE = semantic-layer-dev-metadata
  KNOWLEDGE_BASE_ID = LCY3OWEUOA


In [6]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    # The deployed function carries the stage infix (e.g. semantic-layer-dev-mcp-tools);
    # try both the bare and the '-dev-' variant, mirroring the auth-stack lookup above.
    cfg = None
    for _fn_name in (f'{PROJECT_NAME}-mcp-tools', f'{PROJECT_NAME}-dev-mcp-tools'):
        try:
            cfg = lam.get_function_configuration(FunctionName=_fn_name)
            break
        except lam.exceptions.ResourceNotFoundException:
            continue
    if cfg is None:
        raise RuntimeError(f'mcp-tools Lambda not found for project {PROJECT_NAME}')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


In [7]:
# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
account_id = identity['Account']
print(f"✓ AWS Credentials Verified:")
print(f"  Profile:   {os.environ.get('AWS_PROFILE', 'default')}")
print(f"  Account:   ...{account_id[-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region:    {region}")

✓ AWS Credentials Verified:
  Profile:   huthmac+demo
  Account:   ...4087
  User/Role: huthmac-Isengard
  Region:    us-east-1


## 2. Resolve the AgentCore Ontology Runtime

In [8]:
ac = _get_stack_outputs('agentcore')
ONTOLOGY_RUNTIME_ARN = ac['OntologyRuntimeArn']
agent_id = ONTOLOGY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ Runtime ARN: {_mask_acct(ONTOLOGY_RUNTIME_ARN)}")
print(f"✓ Agent ID:    {agent_id}")

✓ Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:********4087:runtime/semantic_layer_dev_ontology-lysryf3ij6
✓ Agent ID:    semantic_layer_dev_ontology-lysryf3ij6


## 3. Discover the Snowflake tables live + define the VKG config

There is no manifest to read: the tables are **not** ingested into this account,
they live in Snowflake and Athena reaches them through the connector. So the
coordinates come from Athena itself.

`list_table_metadata` is used rather than `DESCRIBE`/Glue because it is the only
metadata call that **works** against a Lambda-backed catalog from the client side:
Glue has no database for `FHIR_R4`, and `DESCRIBE "FHIR_R4"."PATIENTS"` is rejected
by Athena at submit time. The same list is reused in §5 as the ground truth for the
schema-introspection guard.

In [9]:
# ── Discover the live Snowflake tables through the Athena connector ──────────
# Both constants are env-overridable so a different Snowflake database/schema can be
# built without editing the notebook — and notebook 18 reads the SAME two names to
# rebuild LAYER_ID identically. Defaults match `cdk/.env` (SNOWFLAKE_DATABASES).
SNOWFLAKE_CATALOG = os.environ.get('SNOWFLAKE_ATHENA_CATALOG', 'snowflake_fhir_db')
SNOWFLAKE_SCHEMA = os.environ.get('SNOWFLAKE_ATHENA_SCHEMA', 'FHIR_R4')

# DETERMINISTIC layer id — notebook 18 derives it from the same two constants.
LAYER_ID = f'vkg-snowflake_{SNOWFLAKE_SCHEMA.lower()}'

athena = session.client('athena', region_name=region)


def snowflake_tables(*, catalog: str, schema: str) -> dict:
    """List the connector's tables and their column types, straight from Athena.

    The only client-side metadata call that works on a Lambda-backed catalog: the
    tables have no Glue entry (their metadata lives in Snowflake, reachable only
    through the connector), and Athena rejects a quoted schema-qualified DESCRIBE
    on a federated catalog at submit time.

    Parameters:
        catalog: The Athena connector catalog name (e.g. snowflake_fhir_db).
        schema: The Snowflake schema, which Athena presents as the database
            (e.g. FHIR_R4).
    Returns:
        {table_name: [(column_name, athena_type), ...]} for every table.
    Raises:
        RuntimeError: The catalog/schema returned no tables. Without a schema there
            is nothing to build a layer over, and a build seeded from an empty table
            list would "succeed" while modelling nothing.
    """
    found: dict = {}
    paginator = athena.get_paginator('list_table_metadata')
    for page in paginator.paginate(CatalogName=catalog, DatabaseName=schema):
        for table in page['TableMetadataList']:
            found[table['Name']] = [
                (c['Name'], c.get('Type', '')) for c in table.get('Columns', [])
            ]
    if not found:
        raise RuntimeError(
            f"Athena returned no tables for {catalog}.{schema}. Check that the "
            f"connector Lambda is deployed and healthy (-c enableSnowflakeConnector"
            f"=true), that the secret is synced (npm run sync:snowflake-secret), and "
            f"that casing_mode=UPPER is still set on the connector Lambda "
            f"(npm run set:snowflake-casing) — a deploy resets its environment."
        )
    return found


SNOWFLAKE_SCHEMA_MAP = snowflake_tables(catalog=SNOWFLAKE_CATALOG, schema=SNOWFLAKE_SCHEMA)
TABLES = sorted(SNOWFLAKE_SCHEMA_MAP)

# Optional smoke-test slice (mirrors nb4's MAX_TABLES): 0 = build every table.
MAX_TABLES = int(os.environ.get('MAX_TABLES', '0'))
if MAX_TABLES > 0:
    TABLES = TABLES[:MAX_TABLES]
    print(f"  ⚠ MAX_TABLES={MAX_TABLES} — building a {len(TABLES)}-table subset "
          f"(set MAX_TABLES=0 to build all {len(SNOWFLAKE_SCHEMA_MAP)})")

print(f"✓ {SNOWFLAKE_CATALOG}.{SNOWFLAKE_SCHEMA} — {len(SNOWFLAKE_SCHEMA_MAP)} table(s) live:")
for _t in sorted(SNOWFLAKE_SCHEMA_MAP):
    _cols = SNOWFLAKE_SCHEMA_MAP[_t]
    _types = sorted({t for _, t in _cols})
    print(f"  {_t:<16} {len(_cols):>2} cols  types={_types}")
print(f"\n✓ LAYER_ID = {LAYER_ID}  (notebook 18 rebuilds this exact id)")

✓ snowflake_fhir_db.FHIR_R4 — 6 table(s) live:
  CONDITIONS        9 cols  types=['timestamp', 'varchar']
  ENCOUNTERS        9 cols  types=['timestamp', 'varchar']
  FHIR_RESOURCES    5 cols  types=['decimal(38,0)', 'timestamp', 'varchar']
  MEDICATIONS       8 cols  types=['timestamp', 'varchar']
  OBSERVATIONS     10 cols  types=['double', 'timestamp', 'varchar']
  PATIENTS         10 cols  types=['date', 'timestamp', 'varchar']

✓ LAYER_ID = vkg-snowflake_fhir_r4  (notebook 18 rebuilds this exact id)


In [10]:
def sf(*, catalog: str, schema: str, table: str) -> dict:
    """Build the data-source dict the ontology agent expects for a federated table.

    The federated analogue of nb13's `s3t()`. The only field that behaves
    differently is `catalogId`: it is the CONNECTOR catalog name, which
    prompt_builder writes into the ontology as
    `hasCatalog "<database>::<catalog>"`. neptune-tools parses that back into
    `ontologyJson.databases[].catalog`, the Ontop translate handler preserves it,
    and the query agent sets it as `QueryExecutionContext.Catalog` — so this one
    string is what makes the query path reach Snowflake instead of the default
    Glue catalog.

    Parameters:
        catalog: The Athena connector catalog name (e.g. snowflake_fhir_db).
        schema: The Snowflake schema = Athena database (e.g. FHIR_R4).
        table: The table name within the schema.
    Returns:
        A data-source dict: databaseName=schema, catalogId=catalog,
        tableId=f"{catalog}::{schema}.{table}".
    """
    return {
        'databaseName': schema,
        'tableName': table,
        'catalogId': catalog,
        # For an S3 Tables layer this is 'AwsDataCatalog'; for a connector the
        # Athena data source IS the connector catalog. Informational only (it
        # becomes a `hasDataSource` triple) — `catalogId` is what routes queries.
        'dataSource': catalog,
        'tableId': f'{catalog}::{schema}.{table}',
    }


_VKG_USE_CASES = (
    'Build an OWL ontology (virtual knowledge graph) over the FHIR R4 clinical '
    'tables in Snowflake so analysts can ask graph-grounded natural-language '
    'questions about patients, encounters, conditions, medications and '
    'observations. Model one owl:Class per table, an owl:DatatypeProperty per '
    'column, and foreign-key relationships (the *_REF columns reference a '
    'patient/encounter row) as owl:ObjectProperty with rdfs:domain/rdfs:range, '
    'each with a business-meaningful rdfs:comment.'
)

# One test case = one Snowflake schema = one VKG layer = one ontology-agent session.
test_cases = [{
    'id': f'{SNOWFLAKE_CATALOG}.{SNOWFLAKE_SCHEMA}',   # scenario id
    'layer_id': LAYER_ID,
    'category': 'snowflake_federated_ontology',
    'catalog': SNOWFLAKE_CATALOG,
    'schema': SNOWFLAKE_SCHEMA,
    'dynamodb_config': {
        'name': f'snowflake-{SNOWFLAKE_SCHEMA.lower()}-ontology',
        'type': 'VKG',
        'dataSourcesDescription': (
            f'{len(TABLES)} FHIR R4 clinical table(s) in the Snowflake schema '
            f'{SNOWFLAKE_SCHEMA}, queried through the Athena federated connector '
            f'catalog {SNOWFLAKE_CATALOG}. The data lives in Snowflake, not in the '
            f'lake — there is no Glue table for these.'
        ),
        'useCasesDescription': _VKG_USE_CASES,
        'dataSources': [sf(catalog=SNOWFLAKE_CATALOG, schema=SNOWFLAKE_SCHEMA, table=t)
                        for t in TABLES],
    },
}]

print(f"✓ Defined {len(test_cases)} Snowflake VKG test case(s):")
for tc in test_cases:
    n = len(tc['dynamodb_config']['dataSources'])
    print(f"  {tc['id']} → layer {tc['layer_id']} ({n} table(s))")

✓ Defined 1 Snowflake VKG test case(s):
  snowflake_fhir_db.FHIR_R4 → layer vkg-snowflake_fhir_r4 (6 table(s))


In [11]:
# Helper to seed a semantic-layer-metadata DynamoDB record (replicates the admin UI flow).
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', 'semantic-layer-metadata')


def seed_test_ontology(ontology_id: str, tables: list, *, name: str = 'vkg-eval-test',
                       ontology_type: str = 'VKG', data_sources_description: str = '',
                       use_cases_description: str = '',
                       created_by: str = 'eval@semantic-layer.local') -> None:
    """Write a DynamoDB v1 record matching the shape created by the admin UI flow.

    Parameters:
        ontology_id: the `id` partition key the agent will read.
        tables: list of full data-source dicts (from `sf()`).
        name: human-readable config name; gets a UTC timestamp suffix.
        ontology_type: 'VKG' for the ontology (virtual knowledge graph) agent.
        data_sources_description: optional free-text description of the sources.
        use_cases_description: optional free-text description of the intended use cases.
        created_by: user/email recorded as creator.
    """
    ddb_table = session.resource('dynamodb').Table(METADATA_TABLE)
    now = datetime.now(timezone.utc).isoformat()
    ts = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    name_with_ts = f"{name}-{ts}"
    data_sources = [
        {
            'databaseName': t['databaseName'], 'tableName': t['tableName'],
            'catalogId': t.get('catalogId', 'AWSDataCatalog'),
            'dataSource': t.get('dataSource', 'AwsDataCatalog'),
            'tableId': t.get('tableId', f"{t.get('catalogId', 'AWSDataCatalog')}::{t['databaseName']}.{t['tableName']}"),
        }
        for t in tables
    ]
    item = {
        'id': ontology_id, 'version': 'v1', 'type': ontology_type, 'name': name_with_ts,
        'status': 'pending', 'configuration': {}, 'dataSources': data_sources,
        'createdAt': now, 'updatedAt': now, 'buildStartedAt': now, 'createdBy': created_by,
    }
    if data_sources_description:
        item['dataSourcesDescription'] = data_sources_description
    if use_cases_description:
        item['useCasesDescription'] = use_cases_description
    ddb_table.put_item(Item=item)
    print(f"✓ Seeded DynamoDB item: {ontology_id} → name='{name_with_ts}' ({len(data_sources)} table(s))")

print("✓ seed_test_ontology defined")

✓ seed_test_ontology defined


## 4. Create the Custom Ontology-Quality Evaluator — server-side, no Lambda

In [12]:
_cp = session.client('bedrock-agentcore-control', region_name=region)
_SUFFIX = uuid.uuid4().hex[:8]
JUDGE_MODEL_ID = 'global.anthropic.claude-sonnet-5'

# A 0.0–1.0 quality scale (continuous), unlike the binary query-agent judges: ontology
# quality is a matter of degree (coverage / modeling / annotation completeness).
_QUALITY_SCALE = {
    'numerical': [
        {'value': 0.0, 'label': 'poor',
         'definition': 'No owl:Class, almost no properties, or no annotations.'},
        {'value': 0.5, 'label': 'partial',
         'definition': 'Class present but many columns unmodeled, some FKs missed, or shallow comments.'},
        {'value': 1.0, 'label': 'good',
         'definition': 'Full class/property coverage, FK ObjectProperties modeled, business-meaningful rdfs:comments.'},
    ]
}

def _create_llm_judge(*, name: str, level: str, instructions: str) -> str:
    """Create an LLM-as-Judge evaluator and return its evaluatorId.

    Parameters:
        name: human-readable evaluator name (a unique suffix is appended).
        level: 'TRACE' or 'SESSION' — the granularity the service scores at.
        instructions: judge prompt; may reference {context}/{actual_tool_trajectory} etc.

    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f"{name}_{_SUFFIX}",
        level=level,
        evaluatorConfig={
            'llmAsAJudge': {
                'instructions': instructions,
                'ratingScale': _QUALITY_SCALE,
                'modelConfig': {
                    'bedrockEvaluatorModelConfig': {
                        'modelId': JUDGE_MODEL_ID,
                        'inferenceConfig': {'maxTokens': 4096},
                    }
                },
            }
        },
    )
    return resp['evaluatorId']


ONTOLOGY_QUALITY_ID = _create_llm_judge(
    name='OntologyQuality',
    level='SESSION',
    instructions=(
        "You are an expert OWL/RDF ontology evaluator scoring a deployed ontology-generation "
        "agent's output for one session covering one or more tables.\n\n"
        "Session context (full conversation, including tool calls and tool results):\n"
        "{context}\n\n"
        "Available tools: {available_tools}\n"
        "Actual tool trajectory: {actual_tool_trajectory}\n\n"
        "In the context, locate:\n"
        "  (a) the OUTPUT of `get_single_table_schema` — the source table columns; and\n"
        "  (b) the ARGUMENTS of `append_nquads` / `save_intermediate_ontology` — the OWL/RDF "
        "N-Quads the agent generated.\n\n"
        "Score the generated ontology on three axes, then return the MEAN as your score:\n"
        "  1. class/property coverage — exactly one owl:Class per table AND an "
        "owl:DatatypeProperty for (nearly) every source column, each with rdfs:domain/rdfs:range.\n"
        "  2. relationship modeling — foreign-key-like columns (here the *_REF columns, which "
        "reference a patient or encounter row) modeled as owl:ObjectProperty with "
        "rdfs:domain/rdfs:range to the related class (or the table genuinely has no FKs and "
        "none are invented).\n"
        "  3. annotation quality — the owl:Class and most properties carry a business-meaningful "
        "rdfs:comment / rdfs:label (not just the column name restated).\n\n"
        "Score 1.0 (good) when all three axes are strong; 0.5 (partial) when coverage is "
        "incomplete, some FKs are missed, or comments are shallow; 0.0 (poor) when there is no "
        "owl:Class, almost no properties, or no annotations. If no N-Quads are present in the "
        "context, score 0.0 and say the generated ontology was not found. Briefly justify the score."
    ),
)

CUSTOM_EVALUATOR_IDS = [ONTOLOGY_QUALITY_ID]
print("✓ Custom server-side evaluator created (LLM-as-Judge, no Lambda):")
print(f"  OntologyQuality (SESSION) : {ONTOLOGY_QUALITY_ID}")

✓ Custom server-side evaluator created (LLM-as-Judge, no Lambda):
  OntologyQuality (SESSION) : OntologyQuality_7b052a72-1bBahR2wxU


## 5. Build the Dataset + agent_invoker

The schema-introspection guard is the load-bearing check in this notebook. A
connector-routing regression does **not** fail the build: Phase 1 loses the real
schema, the agent falls back to writing an ontology from its own knowledge of FHIR,
and the job still reports `status='completed'`. Such a layer scores like a real one
and measures nothing.

Ground truth here is **Athena**, not Glue: the nb13 version of this guard reads
`glue.get_tables(CatalogId=..., DatabaseName=...)`, which cannot work for a
federated catalog — there is no Glue database to read. `list_table_metadata`
(already fetched in §3) replaces it.

In [13]:
# Per-schema VKG-build invoker + dataset (one scenario).
#
# The ontology agent is ASYNCHRONOUS (background thread + DynamoDB progress), so
# the invoker must: (1) seed a fresh DynamoDB VKG config under the DETERMINISTIC
# layer id (so nb18 can address it), (2) invoke the runtime with the
# framework-managed session_id (so the batch service can correlate spans),
# (3) BLOCK by polling DynamoDB until status is completed/failed, and (4) return
# the build summary. Mirrors nb13's agent_invoker.
ddb_resource = session.resource('dynamodb')
ddb_table = ddb_resource.Table(METADATA_TABLE)

SCENARIO_RUNS: dict = {}                       # scenario_id -> build record
BUILT_LAYER_IDS: list = []                     # deterministic layer ids
_TC_BY_SCENARIO = {tc['id']: tc for tc in test_cases}


def _server_side_seconds(*, item: dict):
    """Compute build latency from the DynamoDB record's own timestamps.

    The agent stamps `buildStartedAt` on the processing transition and
    `completedAt` on completion. Returns None if either is missing/unparseable
    (we never fabricate a value).

    Parameters:
        item: the final DynamoDB item for the ontology build job.
    Returns:
        elapsed seconds as float, or None if timestamps are unavailable.
    """
    started_raw = item.get('buildStartedAt')
    completed_raw = item.get('completedAt')
    if not started_raw or not completed_raw:
        return None
    try:
        return (datetime.fromisoformat(completed_raw)
                - datetime.fromisoformat(started_raw)).total_seconds()
    except (TypeError, ValueError):
        return None


def _federated_column_types(*, schema_map: dict) -> list:
    """Return every column type in the federated schema, as guard ground truth.

    The federated replacement for nb13's `_glue_column_types`. Glue cannot serve
    this: a connector table has no Glue entry at all, so `get_tables` would either
    fail or return nothing and the guard would have no ground truth to judge
    against. The types come from the `list_table_metadata` call in §3, so this
    costs no extra API call.

    Parameters:
        schema_map: {table: [(column, athena_type), ...]} from `snowflake_tables`.
    Returns:
        list[str]: Lowercased column types across all tables, duplicates retained
            (callers only test membership/uniqueness).
    Raises:
        RuntimeError: The map carries no columns — the guard cannot verify anything
            without ground truth, so it fails loudly rather than passing.
    """
    types = [t.lower() for cols in schema_map.values() for _, t in cols]
    if not types:
        raise RuntimeError(
            f"No column types were discovered for {SNOWFLAKE_CATALOG}."
            f"{SNOWFLAKE_SCHEMA}, so the schema-introspection guard cannot verify "
            f"the built ontology. Re-run the discovery cell in section 3."
        )
    return types


def _assert_schema_introspected(*, layer_id: str, s3_client, bucket: str,
                                schema_map: dict) -> None:
    """Fail loudly if a completed ontology was modelled from memory, not from Athena.

    Parameters:
        layer_id: The ontology layer id (e.g. vkg-snowflake_fhir_r4).
        s3_client: A boto3 S3 client used to read the built ontology.nq.
        bucket: The artifacts bucket holding ontologies/<layer_id>/ontology.nq.
        schema_map: The live federated schema from `snowflake_tables`, which is the
            ground truth the range check is judged against.
    Returns:
        None. Raises instead of returning a status, so callers cannot ignore it.
    Raises:
        RuntimeError: The ontology admits it could not introspect the schema, or it
            flattens to all-xsd:string ranges while the real schema has non-string
            columns, or ontology.nq is unreadable.
    """
    key = f"ontologies/{layer_id}/ontology.nq"
    try:
        body = s3_client.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    except Exception as err:  # noqa: BLE001
        raise RuntimeError(
            f"{layer_id} reported completed but s3://{bucket}/{key} is unreadable "
            f"({err}). A completed build must have produced a consolidated ontology."
        ) from err

    # Signal 1: the agent's own admission. Unambiguous — fires on its own.
    if "could not be introspected" in body:
        raise RuntimeError(
            f"{layer_id} was built WITHOUT reading the real schema: ontology.nq "
            f"contains the agent's own 'could not be introspected' admission. For a "
            f"FEDERATED source the usual cause is not Lake Formation but catalog "
            f"ROUTING: Athena rejects `DESCRIBE \"{SNOWFLAKE_SCHEMA}\".\"<table>\"` on "
            f"any Lambda-backed catalog, so the ontology agent must route this "
            f"catalog to the `SELECT * ... LIMIT 0` metadata path (it keys off the "
            f"catalog's registered Type == 'LAMBDA'). If the deployed container "
            f"predates that fix, or its role lacks athena:GetDataCatalog and "
            f"lambda:InvokeFunction on the connector, Phase 1 sees no schema and the "
            f"ontology is written from the LLM's memory of FHIR. Check the ontology "
            f"runtime's logs for 'Routing' and redeploy the agentcore stack. Refusing "
            f"to evaluate a guessed ontology."
        )

    # Signal 2: ranges that lost type information the REAL schema actually has.
    # An all-string schema is legitimate (bird_toxicology is one), so this compares
    # against ground truth rather than treating all-string ranges as proof.
    ranges = [
        line.split("XMLSchema#")[1].split(">")[0]
        for line in body.splitlines()
        if "rdf-schema#range" in line and "XMLSchema#" in line
    ]
    live_types = _federated_column_types(schema_map=schema_map)
    live_has_non_string = any(t not in ("string", "varchar") for t in live_types)
    if len(ranges) > 1 and set(ranges) == {"string"} and live_has_non_string:
        raise RuntimeError(
            f"{layer_id}: all {len(ranges)} property ranges are xsd:string, but the "
            f"live {SNOWFLAKE_CATALOG}.{SNOWFLAKE_SCHEMA} schema has non-string "
            f"columns ({sorted(set(live_types))}). The ontology therefore did NOT "
            f"come from the live schema — see the routing note above, then rebuild. "
            f"Refusing to evaluate a guessed ontology."
        )

    # Signal 3: the tables that were actually modelled. A federated build that
    # silently saw nothing can still emit a plausible ontology, so check that every
    # table we seeded appears in it. Case-insensitive: the agent may lower-case
    # class names even though Snowflake reports them upper-case.
    body_lower = body.lower()
    missing = [t for t in schema_map if t.lower() not in body_lower]
    if missing:
        raise RuntimeError(
            f"{layer_id}: {len(missing)} seeded table(s) do not appear anywhere in "
            f"ontology.nq: {missing}. The build reported completed while modelling "
            f"only part of the schema — check the ontology runtime logs for per-table "
            f"Phase 1 failures before querying this layer."
        )
    print(
        f"  ✓ schema introspection verified: {len(ranges)} typed range(s), "
        f"types={sorted(set(ranges))}; live types={sorted(set(live_types))}; "
        f"all {len(schema_map)} table(s) present"
    )


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the deployed ontology-generation agent for the Snowflake schema and
    block until the async build finishes (so its spans exist for batch discovery).

    BatchEvaluationRunner supplies invoker_input.payload (Turn.input = the scenario
    id) and invoker_input.session_id (framework session id).
    """
    scenario_id = (invoker_input.payload if isinstance(invoker_input.payload, str)
                   else json.dumps(invoker_input.payload))
    tc = _TC_BY_SCENARIO[scenario_id]
    cfg = tc['dynamodb_config']
    tables = cfg['dataSources']
    num_tables = len(tables)
    layer_id = tc['layer_id']            # DETERMINISTIC — nb18 reuses this exact id

    print(f"\n=== scenario '{scenario_id}' -> layer_id={layer_id} "
          f"({num_tables} tables) ===")
    print(f"  runtimeSessionId = {invoker_input.session_id}")

    seed_test_ontology(
        layer_id, tables, name=cfg['name'], ontology_type=cfg.get('type', 'VKG'),
        data_sources_description=cfg.get('dataSourcesDescription', ''),
        use_cases_description=cfg.get('useCasesDescription', ''),
    )

    _invoke_runtime(
        ONTOLOGY_RUNTIME_ARN,
        invoker_input.session_id,
        json.dumps({'id': layer_id}).encode('utf-8'),
    )

    # Block until the background thread reports completion. Budget = fixed
    # refinement headroom + 150s per table for the incremental pass; these ADD (the
    # build runs one LLM pass per table and THEN a holistic refinement pass), so
    # they must not be combined with max(). See nb13 for the measurement this came
    # from. A federated Phase 1 is slightly SLOWER per table than an S3 Tables one:
    # each schema read is a full Athena query through the connector Lambda rather
    # than a Glue API call.
    _poll_floor = int(os.environ.get('BUILD_POLL_FLOOR_SECS', '3600'))
    max_wait_secs = _poll_floor + 150 * num_tables
    start = datetime.now()
    final_item, final_status = {}, 'processing'
    print(f"  Polling DynamoDB for completion (max {max_wait_secs}s)...")
    while (datetime.now() - start).total_seconds() < max_wait_secs:
        time.sleep(30)
        final_item = ddb_table.get_item(
            Key={'id': layer_id, 'version': 'v1'}).get('Item', {})
        final_status = final_item.get('status', 'unknown')
        phase = final_item.get('phase', '')
        pct = final_item.get('progressPercent', '')
        suffix = f" — phase={phase} {pct}%" if phase else ""
        print(f"    [{(datetime.now()-start).total_seconds():.0f}s] "
              f"status={final_status}{suffix}")
        if final_status in ('completed', 'failed'):
            break

    summary = final_item.get('summary') or final_item.get('error', '') or f"status={final_status}"
    build_seconds = _server_side_seconds(item=final_item)

    # Verify the ontology was built from the REAL schema before recording success.
    if final_status == 'completed':
        _assert_schema_introspected(
            layer_id=layer_id,
            s3_client=session.client('s3', region_name=region),
            bucket=os.environ['ARTIFACTS_BUCKET'],
            schema_map={t: SNOWFLAKE_SCHEMA_MAP[t] for t in TABLES},
        )

    SCENARIO_RUNS[scenario_id] = {
        'layer_id': layer_id,
        'session_id': invoker_input.session_id,
        'status': final_status,
        'num_tables': num_tables,
        'tables': [t['tableName'] for t in tables],
        'summary': summary,
        'build_seconds': round(build_seconds, 1) if build_seconds is not None else None,
    }
    if final_status == 'completed' and layer_id not in BUILT_LAYER_IDS:
        BUILT_LAYER_IDS.append(layer_id)
    lat = f"{build_seconds:.1f}s" if build_seconds is not None else 'n/a'
    print(f"  {'✓' if final_status == 'completed' else '✗'} {final_status} — {summary}")
    print(f"     latency(server)={lat}")
    return AgentInvokerOutput(agent_output=summary)


# One scenario. assertions / expected_trajectory are the ground truth the
# SESSION/span built-in evaluators score against (identical to nb13's, plus the
# federated-source assertion).
dataset = Dataset(
    scenarios=[
        PredefinedScenario(
            scenario_id=tc['id'],
            turns=[Turn(input=tc['id'])],     # payload routes the invoker to this tc
            expected_trajectory=[
                'get_single_table_schema',
                'append_nquads',
                'save_intermediate_ontology',
                'update_progress',
            ],
            assertions=[
                "The agent generated an OWL/RDF ontology (owl:Class + owl:DatatypeProperty with rdfs:comment) for every input table.",
                "The table schemas were read from the live federated source, not recalled from the model's knowledge of FHIR.",
                "Foreign-key relationships (the *_REF columns) were modeled as owl:ObjectProperty where applicable.",
                "Per-table N-Quad fragments were persisted via save_intermediate_ontology.",
                "The final DynamoDB status is 'completed'.",
            ],
        )
        for tc in test_cases
    ]
)
print(f"✓ Dataset built: {len(dataset.scenarios)} scenario(s)")

✓ Dataset built: 1 scenario(s)


## 6. Run the Batch Evaluation (server-side, two-phase + resumable)

In [14]:
import uuid as _uuid
from datetime import datetime, timezone

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

ALL_EVALUATOR_IDS = [
    'Builtin.GoalSuccessRate',        # SESSION — assertions
    *CUSTOM_EVALUATOR_IDS,            # custom SESSION ontology-quality judge
]

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=180,  # ontology spans land seconds before the DDB flip
)

# Recovery file: Phase 1 writes the completed session(s) here; Phase 2 reads them.
# Survives a kernel restart, so the long two-phase ontology build never has to be
# re-run just to (re)submit or re-poll the batch.
SESSIONS_FILE = "../data/eval/results/snowflake_vkg_gen_sessions_latest.json"
os.makedirs("../data/eval/results", exist_ok=True)

batch_runner = BatchEvaluationRunner(region=region)

# ── Phase 1: run each scenario's invocation ourselves, then PERSIST ──────────
collected_sessions = []   # sessionId / testScenarioId / window / groundTruth
for scenario in dataset.scenarios:
    sid = f"{scenario.scenario_id}-{_uuid.uuid4()}"
    start_time = datetime.now(timezone.utc)
    print(f"\n▶ Phase 1: scenario '{scenario.scenario_id}' (session_id={sid})")
    for turn in scenario.turns:
        agent_invoker(AgentInvokerInput(payload=turn.input, session_id=sid))
    end_time = datetime.now(timezone.utc)
    # Reuse the SDK's own ground-truth transform so assertions / expected_trajectory
    # are encoded identically to run_dataset_evaluation (no drift).
    ground_truth = batch_runner._transform_ground_truth(scenario)
    collected_sessions.append({
        "sessionId": sid,
        "testScenarioId": scenario.scenario_id,
        "startTime": start_time.isoformat(),
        "endTime": end_time.isoformat(),
        "groundTruth": ground_truth,
    })
    print(f"  ✓ session captured: window=[{start_time.isoformat()}, {end_time.isoformat()}]")

with open(SESSIONS_FILE, "w") as f:
    json.dump(
        {
            "agent_id": agent_id,
            "runtime_arn": _mask_acct(ONTOLOGY_RUNTIME_ARN),
            "service_name": SERVICE_NAME,
            "log_groups": [SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
            "evaluator_ids": ALL_EVALUATOR_IDS,
            "sessions": collected_sessions,
            "scenario_runs": SCENARIO_RUNS,
        },
        f, indent=2, default=str,
    )
print(f"\n✓ Phase 1 complete — {len(collected_sessions)} session(s) persisted to {SESSIONS_FILE}")

# Fail loudly on any build that did not reach `completed`. Without this the
# notebook exits 0 having built nothing, BUILT_LAYER_IDS stays empty, and nb18
# scores an empty layer set while still reporting success.
_incomplete = {
    _sid: _rec.get('status')
    for _sid, _rec in SCENARIO_RUNS.items()
    if _rec.get('status') != 'completed'
}
if _incomplete:
    raise RuntimeError(
        f"{len(_incomplete)} of {len(SCENARIO_RUNS)} layer build(s) did not reach "
        f"status='completed': {_incomplete}. The poll budget may have expired mid-"
        f"refinement (raise BUILD_POLL_FLOOR_SECS) or the agent genuinely failed — "
        f"check the DynamoDB item's `error`/`phase` fields and the runtime logs. "
        f"Refusing to continue: evaluating an incomplete layer produces scores that "
        f"look real but measure nothing."
    )
if not BUILT_LAYER_IDS:
    raise RuntimeError(
        "No layer ids were built, yet every scenario reported completed — this means "
        "the scenario set itself was empty. Check the section-3 table discovery."
    )
print(f"✓ All {len(SCENARIO_RUNS)} build(s) reached status='completed': {BUILT_LAYER_IDS}")

SERVICE_NAME : semantic_layer_dev_ontology-lysryf3ij6.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology-lysryf3ij6-DEFAULT



▶ Phase 1: scenario 'snowflake_fhir_db.FHIR_R4' (session_id=snowflake_fhir_db.FHIR_R4-5d2a3bcd-94bd-4614-8890-ee64e6508f84)

=== scenario 'snowflake_fhir_db.FHIR_R4' -> layer_id=vkg-snowflake_fhir_r4 (6 tables) ===
  runtimeSessionId = snowflake_fhir_db.FHIR_R4-5d2a3bcd-94bd-4614-8890-ee64e6508f84
✓ Seeded DynamoDB item: vkg-snowflake_fhir_r4 → name='snowflake-fhir_r4-ontology-20260813-030435' (6 table(s))


  Polling DynamoDB for completion (max 4500s)...


    [30s] status=processing — phase=incremental %


    [60s] status=processing — phase=incremental %


    [90s] status=processing — phase=incremental %


... (65 outputs left out)

    [2071s] status=processing — phase=refinement 95%


    [2101s] status=processing — phase=refinement 95%


    [2131s] status=processing — phase=refinement 95%


    [2161s] status=completed — phase=refinement_complete 100%
  ✓ schema introspection verified: 57 typed range(s), types=['date', 'dateTime', 'double', 'string']; live types=['date', 'decimal(38,0)', 'double', 'timestamp', 'varchar']; all 6 table(s) present
  ✓ completed — status=completed
     latency(server)=2148.9s
  ✓ session captured: window=[2026-08-13T03:04:35.354900+00:00, 2026-08-13T03:40:46.743231+00:00]

✓ Phase 1 complete — 1 session(s) persisted to ../data/eval/results/snowflake_vkg_gen_sessions_latest.json
✓ All 1 build(s) reached status='completed': ['vkg-snowflake_fhir_r4']


In [15]:
# ── Phase 2 — Submit + poll the batch (idempotent, re-runnable) ──────────────
# Submits ONE StartBatchEvaluation job over the session(s) captured in Phase 1 and
# polls to completion. SAFE TO RE-RUN ON ITS OWN — it reloads the sessions from
# SESSIONS_FILE if they aren't in memory (e.g. after a kernel restart), so a
# Ctrl-C, poll timeout, or FAILED status never forces a re-invocation of the long
# ontology build.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationResult,
    BatchEvaluationSummary,
    CloudWatchOutputDataConfig,
)

POLLING_TIMEOUT_SECONDS = 5400
POLLING_INTERVAL_SECONDS = 30

# Re-runnable: prefer in-memory sessions from Phase 1, else reload from disk.
if "collected_sessions" not in dir() or not collected_sessions:
    with open(SESSIONS_FILE) as f:
        _saved = json.load(f)
    collected_sessions = _saved["sessions"]
    agent_id = _saved["agent_id"]
    ONTOLOGY_RUNTIME_ARN = _saved["runtime_arn"]
    SERVICE_NAME = _saved["service_name"]
    SPANS_LOG_GROUP, RUNTIME_LOG_GROUP = _saved["log_groups"]
    ALL_EVALUATOR_IDS = _saved["evaluator_ids"]
    SCENARIO_RUNS = _saved.get("scenario_runs", {})
    print(f"✓ Reloaded {len(collected_sessions)} session(s) from {SESSIONS_FILE} (Phase 1 not in memory)")

if "batch_runner" not in dir():
    batch_runner = BatchEvaluationRunner(region=region)
if "batch_data_source" not in dir():
    batch_data_source = CloudWatchDataSourceConfig(
        service_names=[SERVICE_NAME],
        log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
        ingestion_delay_seconds=180,
    )

# Parse the persisted session windows back into datetimes.
_session_ids = [s["sessionId"] for s in collected_sessions]
_starts = [datetime.fromisoformat(s["startTime"]) for s in collected_sessions]
_ends = [datetime.fromisoformat(s["endTime"]) for s in collected_sessions]
_session_metadata = [
    {
        "sessionId": s["sessionId"],
        "testScenarioId": s["testScenarioId"],
        **({"groundTruth": {"inline": s["groundTruth"]}} if s.get("groundTruth") else {}),
    }
    for s in collected_sessions
]

batch_name = f"snowflake_vkg_gen_batch_{uuid.uuid4().hex[:8]}"
print(f"Batch name : {batch_name}")
print(f"Evaluators : {ALL_EVALUATOR_IDS}")
print(f"Sessions   : {len(_session_ids)}")
print("Waiting for span ingestion, then submitting StartBatchEvaluation...\n")

# 1) Wait for CloudWatch span ingestion (same hook run_dataset_evaluation calls).
batch_data_source.pre_evaluation_run_hook()

# 2) Submit the batch against the EXISTING sessions (no agent invocation here).
_start_response = batch_runner.data_plane_client.start_batch_evaluation(
    batchEvaluationName=batch_name,
    evaluators=[{"evaluatorId": eid} for eid in ALL_EVALUATOR_IDS],
    dataSourceConfig=batch_data_source.to_data_source_config(
        _session_ids, min(_starts), max(_ends),
    ),
    evaluationMetadata={"sessionMetadata": _session_metadata},
    description="Server-side built-in + custom ontology-quality eval of the Snowflake VKG build.",
)
_batch_id = _start_response["batchEvaluationId"]
_batch_arn = _start_response["batchEvaluationArn"]
print(f"✓ Started batch: {_batch_id}")

# 3) Poll to a terminal state via the SDK's own poll loop.
_response = batch_runner._poll_for_results(
    _batch_id, POLLING_TIMEOUT_SECONDS, POLLING_INTERVAL_SECONDS,
)

# 4) Assemble a BatchEvaluationResult identically to run_dataset_evaluation — a
#    FAILED terminal status still binds batch_result with whatever results exist.
_eval_results = (
    BatchEvaluationSummary.model_validate(_response["evaluationResults"])
    if "evaluationResults" in _response else None
)
_output_data_config = None
if "outputConfig" in _response:
    _odc = _response["outputConfig"].get("cloudWatchConfig")
    if _odc:
        _output_data_config = CloudWatchOutputDataConfig(
            log_group_name=_odc["logGroupName"],
            log_stream_name=_odc["logStreamName"],
        )

batch_result = BatchEvaluationResult(
    batch_evaluation_id=_batch_id,
    batch_evaluation_arn=_batch_arn,
    batch_evaluation_name=_response["batchEvaluationName"],
    status=_response["status"],
    created_at=_response["createdAt"],
    description=_response.get("description"),
    evaluation_results=_eval_results,
    error_details=_response.get("errorDetails"),
    output_data_config=_output_data_config,
)
print(f"\n✓ Batch status: {batch_result.status}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")

Batch name : snowflake_vkg_gen_batch_748e749e
Evaluators : ['Builtin.GoalSuccessRate', 'OntologyQuality_7b052a72-1bBahR2wxU']
Sessions   : 1
Waiting for span ingestion, then submitting StartBatchEvaluation...



✓ Started batch: snowflake_vkg_gen_batch_748e749e-2c67267dc5



✓ Batch status: COMPLETED


## 7. Results — aggregate scores, per-scenario detail

In [16]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"ARN      : {_mask_acct(batch_result.batch_evaluation_arn)}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores ──
agg_rows = []
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (f"{stats.average_score:.3f}"
               if stats and stats.average_score is not None else None)
        agg_rows.append({
            'Evaluator': es.evaluator_id or 'unknown',
            'AvgScore': avg,
            'Evaluated': es.total_evaluated or 0,
            'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

_EVAL_NAME = {ONTOLOGY_QUALITY_ID: 'OntologyQuality'}

# Map framework session id -> scenario for the per-scenario join.
_SCENARIO_BY_SESSION = {r['session_id']: sid for sid, r in SCENARIO_RUNS.items()}

# ── 2. Per-scenario evaluator events (retry: the output stream lags COMPLETED) ──
events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-scenario events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    event_rows.append({
        'scenario_id': _SCENARIO_BY_SESSION.get(sess, sess),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': (str(_event_field(e, 'gen_ai.evaluation.explanation') or '')),
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-scenario evaluator events: {len(df_events)}")

# ── 3. Per-scenario build metadata ──
df_runs = pd.DataFrame(list(SCENARIO_RUNS.values()))

_lat = [r['build_seconds'] for r in SCENARIO_RUNS.values() if r.get('build_seconds') is not None]
perf = {
    'scenarios': len(SCENARIO_RUNS),
    'avg_build_seconds': round(sum(_lat) / len(_lat), 1) if _lat else None,
    'total_build_seconds': round(sum(_lat), 1) if _lat else None,
}
print("\n── Build latency (the agent doing the work) ──")
print(f"  Avg server-side latency : {perf['avg_build_seconds']}s  (n={len(_lat)})")

# ── Persist everything ──
combined_file = f"../data/eval/results/snowflake_vkg_gen_batch_eval_{timestamp}.json"
combined = {
    'source': 'snowflake_federated',
    'catalog': SNOWFLAKE_CATALOG,
    'schema': SNOWFLAKE_SCHEMA,
    'agent_id': agent_id,
    'runtime_arn': _mask_acct(ONTOLOGY_RUNTIME_ARN),
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'OntologyQuality': ONTOLOGY_QUALITY_ID},
    'aggregate_summaries': agg_rows,
    'per_scenario_events': event_rows,
    'scenario_runs': SCENARIO_RUNS,
    'performance': perf,
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores ===")
display(df_agg)
print("=== Per-scenario evaluator scores ===")
display(df_events)
print("=== Per-scenario build metadata ===")
display(df_runs)

Batch ID : snowflake_vkg_gen_batch_748e749e-2c67267dc5
ARN      : arn:aws:bedrock-agentcore:us-east-1:********4087:batch-evaluate/snowflake_vkg_gen_batch_748e749e-2c67267dc5
Status   : COMPLETED

Sessions: completed=1 failed=0 total=1



Per-scenario evaluator events: 2

── Build latency (the agent doing the work) ──
  Avg server-side latency : 2148.9s  (n=1)

✓ Wrote ../data/eval/results/snowflake_vkg_gen_batch_eval_20260813_034517.json

=== Aggregate per-evaluator scores ===


,Evaluator,AvgScore,Evaluated,Failed
0,Builtin.GoalSuccessRate,1.000,1,0
1,OntologyQuality_7b052a72-1bBahR2wxU,1.000,1,0


=== Per-scenario evaluator scores ===


,scenario_id,evaluator,score,label,explanation
0,snowflake_fhir_db.FHIR_R4,Builtin.GoalSuccessRate,1.0,Yes,"The agent's task was specifically for PATIENTS table only (Phase 2): call persist_file_to_neptune for PATIENTS, verify success, then call update_glue_metadata_from_ontology for PATIENTS. Looking at the tool execution history, the agent did call persist_file_to_neptune for PATIENTS (tooluse_RM4Ba6joPUoswKXcbQb3LT) which returned success with 80 triples persisted, and then called update_glue_metadata_from_ontology for PATIENTS (tooluse_m37cNJwwRQV6HwVgwjdMdC) which returned success with columns_updated=0 (expected behavior for federated tables). The agent's final response correctly summarizes: 'Table PATIENTS — ObjectProperties added: 0 — Neptune status: success (80 triples persisted)'. However, the agent also processed many other tables (CONDITIONS, ENCOUNTERS, FHIR_RESOURCES, MEDICATIONS, OBSERVATIONS) which the prompt explicitly said NOT to do ('DO NOT read or process any other table'). The success assertions focus on ontology generation, schema reading, FK modeling, save_intermediate_ontology, and DynamoDB status - all of which were completed. But the agent violated the explicit constraint of processing only PATIENTS. Despite this violation, the PATIENTS-specific steps were completed correctly."
1,snowflake_fhir_db.FHIR_R4,OntologyQuality_7b052a72,1.0,good,"For the PATIENTS table specifically: get_single_table_schema returned 11 columns (id, family_name, given_name, gender, birth_date, city, state, postal_code, last_updated, resource, partition). The append_nquads calls define exactly one owl:Class (Patient) with a rich, business-meaningful rdfs:comment, businessPurpose, businessConcepts, referenceTables, commonQueryPatterns, sampleData and notes annotations. All 11 columns are modeled as owl:DatatypeProperty with rdfs:domain, rdfs:range, rdfs:label, and detailed rdfs:comment (e.g., birth_date, gender, city, state, postal_code, resource, partition all have specific, non-generic comments). Coverage is complete (axis 1 = strong) and annotation quality is high (axis 3 = strong). Regarding relationship modeling (axis 2): PATIENTS has no FK-like *_REF columns (it is the root/parent entity referenced by other tables), and the session context explicitly states 'FK RELATIONSHIPS: none identified for this table.' The agent correctly did not invent any FK ObjectProperties for PATIENTS itself — the inbound FKs (patient_ref from CONDITIONS, ENCOUNTERS, MEDICATIONS, OBSERVATIONS) were correctly modeled as ObjectProperties on those other tables with range Patients, which is appropriate since PATIENTS is the target, not the source, of those relationships. This satisfies the axis 2 condition of 'table genuinely has no FKs and none are invented.' All three axes are strong for this table, and the persist_file_to_neptune step confirms 80 triples were successfully written to Neptune with no other tables' data mixed in, matching the reported ObjectProperties added: 0 (correct, since PATIENTS has no outbound FKs)."


=== Per-scenario build metadata ===


,layer_id,session_id,status,num_tables,tables,summary,build_seconds
0,vkg-snowflake_fhir_r4,snowflake_fhir_db.FHIR_R4-5d2a3bcd-94bd-4614-8890-ee64e6508f84,completed,6,"[CONDITIONS, ENCOUNTERS, FHIR_RESOURCES, MEDICATIONS, OBSERVATIONS, PATIENTS]",status=completed,2148.9


## 8. Store the Layer ID for Notebook 18

In [17]:
# Persist the built layer id so 18_snowflake_query_eval can bind to it. Written
# both to the IPython %store DB (convenient in-session) and to a small JSON file
# (survives across kernels / notebooks). nb18 also rebuilds the id from
# SNOWFLAKE_CATALOG / SNOWFLAKE_SCHEMA and verifies it in DynamoDB, so this file is
# a convenience, not a hard dependency.
snowflake_vkg_layer_ids = list(BUILT_LAYER_IDS)
%store snowflake_vkg_layer_ids

layers_file = "../data/eval/results/snowflake_vkg_built_layers.json"
os.makedirs("../data/eval/results", exist_ok=True)
with open(layers_file, 'w') as f:
    json.dump({
        'source': 'snowflake_federated',
        'catalog': SNOWFLAKE_CATALOG,
        'schema': SNOWFLAKE_SCHEMA,
        'ontology_runtime_arn': _mask_acct(ONTOLOGY_RUNTIME_ARN),
        'agent_id': agent_id,
        'layer_id_convention': 'vkg-snowflake_<schema lower-cased>',
        'built_layer_ids': snowflake_vkg_layer_ids,
        'tables': TABLES,
        'scenario_runs': SCENARIO_RUNS,
    }, f, indent=2, default=str)

print("✓ Stored for notebook 18:")
print(f"  snowflake_vkg_layer_ids = {snowflake_vkg_layer_ids}")
print(f"  layers file             = {layers_file}")

Stored 'snowflake_vkg_layer_ids' (list)
✓ Stored for notebook 18:
  snowflake_vkg_layer_ids = ['vkg-snowflake_fhir_r4']
  layers file             = ../data/eval/results/snowflake_vkg_built_layers.json


## Summary

This notebook builds a **VKG (OWL) semantic layer over live Snowflake tables**
reached through the Athena federated connector, by driving the deployed
ontology-generation agent, and scores the generated ontology **entirely
server-side** via AgentCore Batch Evaluations.

### Pipeline
1. **Discover** the tables live from Athena `list_table_metadata` on
   `snowflake_<db>` / `<schema>` — there is no manifest and no Glue database,
   because the data is in Snowflake.
2. **Build**: seed a DynamoDB VKG config under the deterministic id
   `vkg-snowflake_<schema>`, invoke the ontology agent, block on the DynamoDB
   build status, then run the **schema-introspection guard** (three signals: the
   agent's own "could not be introspected" admission, all-`xsd:string` ranges
   against a live schema that has non-string columns, and any seeded table missing
   from `ontology.nq`).
3. **Evaluators** (server-side): `Builtin.GoalSuccessRate` (assertions) + a custom
   `OntologyQuality` SESSION LLM-as-Judge.
4. **Two-phase, resumable batch**: Phase 1 invokes + persists the completed
   session; Phase 2 submits one `StartBatchEvaluation` and polls (re-runnable
   without re-invoking the long build).
5. **Results** → `../data/eval/results/snowflake_vkg_gen_batch_eval_*.json`.

### What a failure here usually means
| Symptom | Cause |
|:--|:--|
| §3 raises "returned no tables" | connector Lambda unhealthy, secret unsynced, or `casing_mode` reset by a deploy (`npm run set:snowflake-casing`) |
| guard raises "could not be introspected" | the ontology agent routed the connector catalog to `DESCRIBE` (pre-fix container) or its role lacks `athena:GetDataCatalog` / `lambda:InvokeFunction` on the connector |
| guard raises "seeded table(s) do not appear" | some Phase-1 table reads failed while the build still completed |
| `status='failed'` | read the DynamoDB item's `error` / `phase`, then the runtime log group |

### Next
`18_snowflake_query_eval.ipynb` binds `ontologyId = "vkg-snowflake_fhir_r4"` and
asks the query agent natural-language questions, checking the answers against
results executed directly on Athena.
